# [UM]-Net — Directed Graph Agent Network

> **[UM] as AI agent. Fire-and-forget directed graph. Multiple [E]'s.**<br>
> *August 4, 2026 — Distributed agent architecture without coordination*

## Generalization

Notebook 03 showed the linear chain:

```text
[E] → h₁ → (UM) → h₂ → (UM) → h₃ → ... → [E]
```

The natural generalization:

```text
[E₁] ──encodings──→ [UM_A] ──encodings──→ [UM_C] ──encodings──→ [E₃]
                         │                    ↑
[E₂] ──encodings──→ [UM_B] ──encodings──────┘

       edges carry materialized encoding collections
       HLLSets are internal machinery within each [UM]
       multiple environments on both sides
```

### What flows between agents: encodings, not HLLSets

HLLSets are the agent's internal bit-level scratchpad — the gate, the LUT
lookup, the cross-validation all operate on bits. But HLLSets are **lossy**
(multiple tokens collide at the same bit position). Passing them between
agents would discard the disambiguation the upstream agent just performed.

**Materialized encodings** preserve the full ordered result. Each agent
re-tokenizes incoming encodings into its own internal HLLSet. Real tokens
(words, characters) are outside the [UM]-net per the black-box principle.

### What makes this work without coordination

| IICA property | Fire-and-forget consequence |
|---------------|----------------------------|
| **Idempotent** | Duplicate encoding → same internal HLLSet → no-op |
| **Immutable** | No version conflicts — HLLSets never change internally |
| **Content-Addressed** | Internal HLLSets self-identify via SHA-1 key |
| **Union is CRDT** | Merge internal HLLSets at confluence without consensus |
| **Commutative** | A ∪ B = B ∪ A — arrival order doesn't matter |
| **Associative** | (A ∪ B) ∪ C = A ∪ (B ∪ C) — any merge topology works |

**Prerequisites:** `hllset-py` built and installed.

---
## 1. Setup and Agent Definition

In [1]:
import sys, re
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Set, Optional
from collections import defaultdict

import hllset_py

def extract_chinese(text): return re.findall(r'[\u4e00-\u9fff]', text)
def make_1grams(text): return extract_chinese(text)
def make_2grams(text):
    c=extract_chinese(text); return [c[i]+c[i+1] for i in range(len(c)-1)]
def make_3grams(text):
    c=extract_chinese(text); return [c[i]+c[i+1]+c[i+2] for i in range(len(c)-2)]

print('Utilities loaded.')

Utilities loaded.


In [2]:
@dataclass
class Agent:
    """An autonomous [UM] agent.
    
    Receives materialized encodings from upstream, re-tokenizes
    them into its internal HLLSet, processes independently, and
    fires its own encodings downstream.
    
    HLLSets are internal machinery — they never leave the agent.
    What flows between agents is the materialized encoding collection.
    """
    name: str
    lut1: hllset_py.TokenLut = field(default_factory=hllset_py.TokenLut)
    lut2: hllset_py.TokenLut = field(default_factory=hllset_py.TokenLut)
    lut3: hllset_py.TokenLut = field(default_factory=hllset_py.TokenLut)
    gate: Optional[hllset_py.HLLSet] = None
    domain: str = ''
    messages_received: int = 0
    messages_sent: int = 0
    
    def seed_vocabulary(self, texts: List[str]):
        chars = set()
        for t in texts: chars.update(extract_chinese(t))
        self.lut1.record_all(sorted(chars))
        for t in texts:
            self.lut1.record_all(make_1grams(t))
            self.lut2.record_all(make_2grams(t))
            self.lut3.record_all(make_3grams(t))
        if chars:
            self.gate = hllset_py.HLLSet.from_tokens(sorted(chars))
    
    def process(self, encodings: str = '') -> Dict:
        """Process incoming encodings. Fire-and-forget: no ACK.
        
        Receives materialized encodings, re-tokenizes into internal
        HLLSet, runs full pipeline, returns new encodings downstream.
        """
        self.messages_received += 1
        
        # Step 1: Re-tokenize encodings into internal HLLSet
        g1 = make_1grams(encodings) if encodings else []
        h_raw = hllset_py.HLLSet.from_tokens(g1) if g1 else hllset_py.HLLSet.from_tokens([])
        
        # Step 2: Gate filter (internal)
        if self.gate is not None:
            h_gated = h_raw.intersection(self.gate)
        else:
            h_gated = h_raw
        
        # Step 3: Materialize from this agent's LUTs (internal)
        mat1 = hllset_py.materialize(h_gated, self.lut1)
        mat2 = hllset_py.materialize(h_gated, self.lut2)
        mat3 = hllset_py.materialize(h_gated, self.lut3)
        
        # Step 4: Cross-validate (internal)
        scores = {}
        for ch in mat1:
            tf = self.lut1.tf(ch)
            sup2 = sum(1 for bg in mat2 if ch in bg)
            sup3 = sum(1 for tg in mat3 if ch in tg)
            scores[ch] = tf + sup2*0.5 + sup3*0.25
        ranked = [c for c,_ in sorted(scores.items(), key=lambda x:-x[1])]
        encodings_out = ''.join(ranked)
        
        self.messages_sent += 1
        
        return {
            'agent': self.name,
            'encodings': encodings_out,
            'hllset_key': h_gated.content_key()[:24],
            'materialized': ranked,
            'mat1_count': len(mat1),
            'mat2_count': len(mat2),
            'mat3_count': len(mat3),
            'gate_popcount': self.gate.popcount() if self.gate else 0,
        }
    
    def summary(self) -> Dict:
        return {
            'name': self.name, 'domain': self.domain,
            'lut1_size': self.lut1.len(), 'lut2_size': self.lut2.len(),
            'lut3_size': self.lut3.len(),
            'gate_bits': self.gate.popcount() if self.gate else 0,
            'msgs_rx': self.messages_received, 'msgs_tx': self.messages_sent,
        }

print('Agent defined: process(encodings) → encodings (fire-and-forget)')

Agent defined: process(encodings) → encodings (fire-and-forget)


---
## 2. Create the Agent Fleet

In [3]:
iching_texts = [
    '乾元亨利貞', '坤元亨利牝馬之貞',
    '屯元亨利貞勿用有攸往利建侯', '蒙亨匪我求童蒙童蒙求我',
    '需有孚光亨貞吉利涉大川', '訟有孚窒惕中吉終凶利見大人不利涉大川',
]
driving_texts = [
    '车辆在十字路口减速慢行', '高速公路上保持安全距离',
    '雨天路滑降低车速', '红灯停车绿灯通行',
    '行人过马路走斑马线', '转弯前打转向灯示意',
    '遇到紧急车辆及时让行', '夜间行车开启近光灯',
    '酒后严禁驾驶车辆', '系好安全带保护生命',
]

agent_iching   = Agent('ICHING',   domain='iching');   agent_iching.seed_vocabulary(iching_texts)
agent_driving  = Agent('DRIVING',  domain='driving');  agent_driving.seed_vocabulary(driving_texts)
agent_combined = Agent('COMBINED', domain='combined'); agent_combined.seed_vocabulary(iching_texts + driving_texts)

all_iching = set(); [all_iching.update(extract_chinese(t)) for t in iching_texts]
tf_ranked = sorted(all_iching, key=lambda c: -agent_iching.lut1.tf(c))
agent_narrow = Agent('NARROW', domain='iching-narrow')
agent_narrow.seed_vocabulary(iching_texts)
agent_narrow.gate = hllset_py.HLLSet.from_tokens(tf_ranked[:10])

fleet = [agent_iching, agent_driving, agent_combined, agent_narrow]
print(f'{"Agent":10s} {"LUT1":5s} {"LUT2":5s} {"LUT3":5s} {"Gate":5s}')
print('-' * 32)
for a in fleet:
    s = a.summary()
    print(f'{s["name"]:10s} {s["lut1_size"]:4d}  {s["lut2_size"]:4d}  {s["lut3_size"]:4d}  {s["gate_bits"]:4d}')

Agent      LUT1  LUT2  LUT3  Gate 
--------------------------------
ICHING       38    50    49    38
DRIVING      68    79    72    68
COMBINED    104   129   121   104
NARROW       38    50    49    10


---
## 3. Fire-and-Forget: Encodings In, Encodings Out

Agent receives encodings, re-tokenizes into internal HLLSet, processes,
returns new encodings. No HLLSet ever crosses an agent boundary.

In [4]:
print('=== FIRE-AND-FORGET: Encodings → Agent → Encodings ===')
print()

text = '元亨利貞车辆在十字路口'
print(f'Input encodings: "{text}"')
print()

# Each agent receives the SAME encodings, re-tokenizes independently
results = {}
for agent in [agent_iching, agent_driving, agent_combined]:
    msg = agent.process(text)
    results[agent.name] = msg
    print(f'  {agent.name:10s}: gate={msg["gate_popcount"]:3d}  '
          f'mat1={msg["mat1_count"]:2d}  mat2={msg["mat2_count"]:2d}  mat3={msg["mat3_count"]:2d}  '
          f'→ "{msg["encodings"]}"')

print()

# CRDT demo: internal HLLSet rebuilt from encodings for merge verification
h_iching   = hllset_py.HLLSet.from_tokens(make_1grams(results['ICHING']['encodings']))
h_driving  = hllset_py.HLLSet.from_tokens(make_1grams(results['DRIVING']['encodings']))
h_combined = hllset_py.HLLSet.from_tokens(make_1grams(results['COMBINED']['encodings']))
merged = h_iching.union(h_driving).union(h_combined)

print(f'Internal HLLSet union (CRDT): pop={merged.popcount()}  key={merged.content_key()[:24]}...')
merged_mat = hllset_py.materialize(merged, agent_combined.lut1)
indiv_union = set()
for name in ['ICHING', 'DRIVING', 'COMBINED']:
    indiv_union.update(results[name]['materialized'])
print(f'Match individual union: {set(merged_mat) == indiv_union}')
print()
print(f'Each agent processed from the same encodings independently.')

=== FIRE-AND-FORGET: Encodings → Agent → Encodings ===

Input encodings: "元亨利貞车辆在十字路口"

  ICHING    : gate= 38  mat1= 4  mat2= 0  mat3= 0  → "利亨貞元"
  DRIVING   : gate= 68  mat1= 7  mat2= 1  mat3= 1  → "车路辆字在口十"
  COMBINED  : gate=104  mat1=11  mat2= 1  mat3= 1  → "利车亨貞路元辆字在口十"

Internal HLLSet union (CRDT): pop=11  key=h:e9fbc404227d37a4d1c017...
Match individual union: True

Each agent processed from the same encodings independently.


---
## 4. Graph Topology and Execution

In [5]:
@dataclass
class AgentGraph:
    """Directed graph of [UM] agents.
    
    Edges carry materialized encodings (fire-and-forget).
    At confluence: incoming encoding strings are concatenated
    and re-tokenized by the receiving agent internally.
    """
    agents: Dict[str, Agent] = field(default_factory=dict)
    edges: List[Tuple[str, str]] = field(default_factory=list)
    
    def add_agent(self, agent: Agent):
        self.agents[agent.name] = agent
    
    def add_edge(self, src: str, dst: str):
        self.edges.append((src, dst))
    
    @property
    def sources(self) -> Set[str]:
        has_incoming = {dst for _, dst in self.edges}
        return set(self.agents) - has_incoming
    
    @property
    def sinks(self) -> Set[str]:
        has_outgoing = {src for src, _ in self.edges}
        return set(self.agents) - has_outgoing
    
    def topology_str(self) -> str:
        return '\n'.join(f'  [{s}] ──enc──→ [{d}]' for s, d in self.edges)
    
    def execute(self, inputs: Dict[str, str]) -> Dict[str, List[Dict]]:
        """Execute graph with encoding inputs at source agents."""
        inbox: Dict[str, List[str]] = defaultdict(list)
        output: Dict[str, List[Dict]] = defaultdict(list)
        
        for agent_name, enc in inputs.items():
            inbox[agent_name].append(enc)
        
        downstream = defaultdict(list)
        for src, dst in self.edges:
            downstream[src].append(dst)
        
        # BFS levels from sources
        levels = {name: 0 for name in self.sources}
        changed = True
        while changed:
            changed = False
            for src, dst in self.edges:
                if src in levels and dst not in levels:
                    preds = [s for s, d in self.edges if d == dst]
                    if all(p in levels for p in preds):
                        levels[dst] = max(levels[p] for p in preds) + 1
                        changed = True
        for name in self.agents:
            if name not in levels:
                levels[name] = 0
        
        max_level = max(levels.values()) if levels else 0
        for level in range(max_level + 1):
            for agent_name in [n for n, l in levels.items() if l == level]:
                agent = self.agents[agent_name]
                incoming = inbox.get(agent_name, [])
                if not incoming:
                    continue
                merged_enc = ''.join(incoming)
                msg = agent.process(merged_enc)
                output[agent_name].append(msg)
                for dst in downstream.get(agent_name, []):
                    inbox[dst].append(msg['encodings'])
        
        return dict(output)

print('AgentGraph: edges carry encodings, confluence concatenates, re-tokenizes internally.')

AgentGraph: edges carry encodings, confluence concatenates, re-tokenizes internally.


---
## 5. Diamond Topology: Fork → Process → Merge

In [6]:
g = AgentGraph()
g.add_agent(agent_combined)
g.add_agent(agent_iching)
g.add_agent(agent_driving)

sink_agent = Agent('SINK', domain='merged')
sink_agent.seed_vocabulary(iching_texts + driving_texts)
g.add_agent(sink_agent)

g.add_edge('COMBINED', 'ICHING')
g.add_edge('COMBINED', 'DRIVING')
g.add_edge('ICHING', 'SINK')
g.add_edge('DRIVING', 'SINK')

print('=== DIAMOND TOPOLOGY ===')
print(g.topology_str())
print(f'Sources: {g.sources}  Sinks: {g.sinks}')
print()

text = '元亨利貞车辆在十字路口减速慢行'
output = g.execute({'COMBINED': text})

for name in ['COMBINED', 'ICHING', 'DRIVING', 'SINK']:
    for msg in output.get(name, []):
        print(f'[{name:10s}]: gate={msg["gate_popcount"]:3d}  '
              f'mat1={msg["mat1_count"]:2d}  mat2={msg["mat2_count"]:2d}  mat3={msg["mat3_count"]:2d}  '
              f'→ "{msg["encodings"]}"')

print()
print(f'SINK concatenated encodings from ICHING + DRIVING, re-tokenized.')

=== DIAMOND TOPOLOGY ===
  [COMBINED] ──enc──→ [ICHING]
  [COMBINED] ──enc──→ [DRIVING]
  [ICHING] ──enc──→ [SINK]
  [DRIVING] ──enc──→ [SINK]
Sources: {'COMBINED'}  Sinks: {'SINK'}

[COMBINED  ]: gate=104  mat1=15  mat2= 2  mat3= 1  → "利车行亨貞路元速辆字减在口慢十"
[ICHING    ]: gate= 38  mat1= 4  mat2= 0  mat3= 0  → "利亨貞元"
[DRIVING   ]: gate= 68  mat1=11  mat2= 2  mat3= 1  → "车行路速辆字减在口慢十"
[SINK      ]: gate=104  mat1=15  mat2= 2  mat3= 1  → "利车行亨貞路元速辆字减在口慢十"

SINK concatenated encodings from ICHING + DRIVING, re-tokenized.


---
## 6. Multi-Environment: Multiple [E] Entry Points

In [7]:
g2 = AgentGraph()
for a in [agent_combined, agent_iching, agent_driving, agent_narrow, sink_agent]:
    g2.add_agent(a)

g2.add_edge('COMBINED', 'ICHING')
g2.add_edge('ICHING', 'SINK')
g2.add_edge('NARROW', 'SINK')
g2.add_edge('DRIVING', 'SINK')

print('=== MULTI-ENVIRONMENT ===')
print(g2.topology_str())
print()

e1 = '元亨利貞'
e2 = '车辆在十字路口'
print(f'[E₁]: "{e1}"  [E₂]: "{e2}"')
print()

output2 = g2.execute({'COMBINED': e1, 'DRIVING': e2, 'NARROW': e1})

for name in ['COMBINED', 'ICHING', 'DRIVING', 'NARROW', 'SINK']:
    for msg in output2.get(name, []):
        print(f'[{name:10s}] gate={msg["gate_popcount"]:3d}  '
              f'mat1={msg["mat1_count"]:2d}  → "{msg["encodings"]}"')

print()
for a in [agent_combined, agent_iching, agent_driving, agent_narrow, sink_agent]:
    s = a.summary()
    print(f'  {s["name"]:10s}: rx={s["msgs_rx"]}  tx={s["msgs_tx"]}')

=== MULTI-ENVIRONMENT ===
  [COMBINED] ──enc──→ [ICHING]
  [ICHING] ──enc──→ [SINK]
  [NARROW] ──enc──→ [SINK]
  [DRIVING] ──enc──→ [SINK]

[E₁]: "元亨利貞"  [E₂]: "车辆在十字路口"

[COMBINED  ] gate=104  mat1= 4  → "利亨貞元"
[ICHING    ] gate= 38  mat1= 4  → "利亨貞元"
[DRIVING   ] gate= 68  mat1= 7  → "车路辆字在口十"
[NARROW    ] gate= 10  mat1= 4  → "利亨貞元"
[SINK      ] gate=104  mat1=11  → "利车亨貞路元辆字在口十"

  COMBINED  : rx=3  tx=3
  ICHING    : rx=3  tx=3
  DRIVING   : rx=3  tx=3
  NARROW    : rx=1  tx=1
  SINK      : rx=2  tx=2


---
## 7. CRDT Merge Properties (Internal Mechanism)

In [8]:
print('=== CRDT MERGE PROPERTIES ===')

t1, t2, t3 = '元亨利貞', '车辆在十字路口', '减速慢行安全距离'

r1 = agent_iching.process(t1)
r2 = agent_driving.process(t2)
r3 = agent_combined.process(t3)
h1 = hllset_py.HLLSet.from_tokens(make_1grams(r1['encodings']))
h2 = hllset_py.HLLSet.from_tokens(make_1grams(r2['encodings']))
h3 = hllset_py.HLLSet.from_tokens(make_1grams(r3['encodings']))

print(f'H1: pop={h1.popcount()}  H2: pop={h2.popcount()}  H3: pop={h3.popcount()}')
print()

m1 = h1.union(h2).union(h3)
m2 = h1.union(h2).union(h3)
print(f'Idempotent:   m1 == m2 → {m1.content_key() == m2.content_key()}')

m_abc = h1.union(h2).union(h3)
m_cba = h3.union(h2).union(h1)
print(f'Commutative:  abc == cba → {m_abc.content_key() == m_cba.content_key()}')

m_ab_c = h1.union(h2).union(h3)
m_a_bc = h1.union(h2.union(h3))
print(f'Associative:  (ab)c == a(bc) → {m_ab_c.content_key() == m_a_bc.content_key()}')

print(f'Monotonic:    H1={h1.popcount()} → H1∪H2={h1.union(h2).popcount()} → H1∪H2∪H3={m1.popcount()}')
print()
print(f'HLLSet union is state-based CRDT. No consensus. No leader election.')

=== CRDT MERGE PROPERTIES ===
H1: pop=4  H2: pop=7  H3: pop=8

Idempotent:   m1 == m2 → True
Commutative:  abc == cba → True
Associative:  (ab)c == a(bc) → True
Monotonic:    H1=4 → H1∪H2=11 → H1∪H2∪H3=19

HLLSet union is state-based CRDT. No consensus. No leader election.


---
## 8. Robustness: Loss, Duplicates, Ordering

In [9]:
print('=== ROBUSTNESS ===')
enc = '元亨利貞车辆'

# Duplicate
m1 = agent_combined.process(enc)
m2 = agent_combined.process(enc)
print(f'DUPLICATE:    same key? {m1["hllset_key"] == m2["hllset_key"]}  (idempotent)')

# Out of order
r_a = agent_iching.process(enc)
r_b = agent_driving.process(enc)
h_a = hllset_py.HLLSet.from_tokens(make_1grams(r_a['encodings']))
h_b = hllset_py.HLLSet.from_tokens(make_1grams(r_b['encodings']))
print(f'OUT OF ORDER: A∪B == B∪A? {h_a.union(h_b).content_key() == h_b.union(h_a).content_key()}')

# Lost (one agent missing, recoverable by union)
r_i = agent_iching.process(enc)
h_i = hllset_py.HLLSet.from_tokens(make_1grams(r_i['encodings']))
mat_i = hllset_py.materialize(h_i, agent_combined.lut1)
r_d = agent_driving.process(enc)
h_both = h_i.union(hllset_py.HLLSet.from_tokens(make_1grams(r_d['encodings'])))
mat_both = hllset_py.materialize(h_both, agent_combined.lut1)
print(f'LOST MSG:     partial={len(mat_i)} chars; full={len(mat_both)} chars (recoverable)')

print(f'\nDuplicates harmless. Order irrelevant. Loss recoverable.')

=== ROBUSTNESS ===
DUPLICATE:    same key? True  (idempotent)
OUT OF ORDER: A∪B == B∪A? True
LOST MSG:     partial=4 chars; full=6 chars (recoverable)

Duplicates harmless. Order irrelevant. Loss recoverable.


---
## 9. [E] → Graph → [E] Feedback Loop

In [10]:
print('=== [E] → GRAPH → [E] LOOP ===')
print()

e1_text = '元亨利貞'
e2_text = '车辆在十字路口'
print(f'[E₁]: "{e1_text}"  [E₂]: "{e2_text}"')
print()

output = g.execute({'COMBINED': e1_text, 'DRIVING': e2_text})

sink_msgs = output.get('SINK', [])
if sink_msgs:
    sink = sink_msgs[0]
    print(f'Graph output [SINK]: "{sink["encodings"]}"')
    if 'ICHING' in output:
        print(f'  ICHING contributed: {output["ICHING"][0]["materialized"]}')
    if 'DRIVING' in output:
        print(f'  DRIVING contributed: {output["DRIVING"][0]["materialized"]}')
    
    h_e1_in = hllset_py.HLLSet.from_tokens(make_1grams(e1_text))
    h_sink = hllset_py.HLLSet.from_tokens(make_1grams(sink['encodings']))
    bss = h_e1_in.bss_inclusion(h_sink)
    print(f'\nFeedback BSS: {bss:.4f} (structural relevance of graph output to [E₁])')
    print()
    print(f'IICA guarantees the feedback is structurally relevant, not random.')

=== [E] → GRAPH → [E] LOOP ===

[E₁]: "元亨利貞"  [E₂]: "车辆在十字路口"

Graph output [SINK]: "利车亨貞路元辆字在口十"
  ICHING contributed: ['利', '亨', '貞', '元']
  DRIVING contributed: ['车', '路', '辆', '字', '在', '口', '十']

Feedback BSS: 0.3636 (structural relevance of graph output to [E₁])

IICA guarantees the feedback is structurally relevant, not random.


---
## Summary

| # | Section | Key result |
|---|---------|------------|
| 1-2 | Agent fleet | 4 agents with independent LUTs and gates |
| 3 | Fire-and-forget | Encodings in → re-tokenize → process → encodings out |
| 4 | Graph topology | AgentGraph with wave execution |
| 5 | Diamond | Fork→process→merge with encoding concatenation |
| 6 | Multi-environment | Two [E] inputs at different entry points |
| 7 | CRDT merge | Internal HLLSet union: idempotent, commutative, associative, monotonic |
| 8 | Robustness | Duplicates harmless, order irrelevant, loss recoverable |
| 9 | [E]→Graph→[E] | BSS feedback relevance measurement |

**What flows between agents:** materialized encodings.
**HLLSets:** internal bit-level machinery (gate, LUT, cross-validation).
**Real tokens:** outside the [UM]-net (black-box principle).
**Shared storage:** IPFS for HLLSet persistence, recovery, and audit.